# Notebook 02 — Trích xuất đặc trưng (Feature Extraction)
**DSP501 — Nhận dạng người nói (Speaker Identification)**

Notebook này giải thích chi tiết quá trình **trích xuất đặc trưng** từ tín hiệu âm thanh — bước quan trọng nhất để máy có thể "nhận ra" giọng nói của từng người.

### Nội dung:
1. **Load & tiền xử lý** một file âm thanh mẫu
2. **Pipeline A** — Đặc trưng miền thời gian cơ bản (Energy, ZCR)
3. **Pipeline B** — Bộ lọc FIR → Pre-emphasis → MFCC (đặc trưng tần số nâng cao)
4. **So sánh trực quan** hai pipeline qua biểu đồ
5. **Xây dựng dataset** đầy đủ cho huấn luyện

### Tại sao cần trích xuất đặc trưng?
Tín hiệu âm thanh thô (raw waveform) có **48,000 mẫu** cho 3 giây — quá nhiều chiều để máy học hiệu quả. Ta cần rút gọn thành một **vector đặc trưng nhỏ** (6 hoặc 26 chiều) mà vẫn giữ được thông tin quan trọng để phân biệt người nói.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import pandas as pd

from preprocess import preprocess
from filter import design_fir, apply_filter
from preemphasis import pre_emphasize
from feature_extraction import extract_mfcc, extract_basic_features, save_features

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

---
## 1. Load và tiền xử lý một file âm thanh mẫu

Trước khi trích xuất đặc trưng, mỗi file audio đều phải qua **4 bước tiền xử lý** (đã làm ở Notebook 01):

| Bước | Hàm | Mục đích |
|------|-----|----------|
| 1 | `load_audio()` | Đọc file WAV, chuyển mono, resample 16kHz |
| 2 | `normalize()` | Chuẩn hóa biên độ về [-1, 1] để các file có cùng mức âm lượng |
| 3 | `trim_silence()` | Cắt bỏ khoảng lặng đầu/cuối (ngưỡng -20dB) |
| 4 | `pad_or_crop()` | Đảm bảo đúng 48,000 mẫu (3 giây × 16,000 Hz) |

Ta sẽ dùng file của **Cuong** (`speaker_08/01.wav`) làm ví dụ minh họa.

In [ ]:
# ── Load 1 file mẫu ──────────────────────────────────────────────
SAMPLE = '../data/raw/speaker_08/01.wav'   # File của Cuong
SR = 16000                                  # Tần số lấy mẫu 16kHz

y_raw, sr = preprocess(SAMPLE, sr=SR)

print(f'Tần số lấy mẫu  : {sr} Hz')
print(f'Số mẫu (samples) : {len(y_raw):,}')
print(f'Thời lượng       : {len(y_raw)/sr:.1f} giây')
print(f'Biên độ min/max  : [{y_raw.min():.3f}, {y_raw.max():.3f}]')

# Vẽ dạng sóng
fig, ax = plt.subplots(figsize=(12, 3))
time = np.arange(len(y_raw)) / sr
ax.plot(time, y_raw, linewidth=0.5, color='steelblue')
ax.set_xlabel('Thời gian (giây)')
ax.set_ylabel('Biên độ')
ax.set_title('Dạng sóng sau tiền xử lý — Cuong (speaker_08/01.wav)')
ax.set_xlim(0, len(y_raw)/sr)
plt.tight_layout()
plt.show()

---
## 2. Pipeline A — Đặc trưng miền thời gian cơ bản (Baseline)

Pipeline A **không sử dụng kỹ thuật DSP nâng cao** nào — chỉ trích xuất các đặc trưng đơn giản trực tiếp từ tín hiệu thô:

| # | Đặc trưng | Ý nghĩa | Công thức |
|---|-----------|---------|-----------|
| 1 | **RMS Energy (mean)** | Năng lượng trung bình — giọng to hay nhỏ | $\sqrt{\frac{1}{N}\sum x[n]^2}$ |
| 2 | **RMS Energy (std)** | Biến thiên năng lượng — giọng đều hay thay đổi | Độ lệch chuẩn của RMS qua các frame |
| 3 | **ZCR (mean)** | Tốc độ tín hiệu đổi dấu — phân biệt giọng trầm/cao | Số lần tín hiệu cắt trục 0 trên giây |
| 4 | **ZCR (std)** | Biến thiên ZCR | Độ lệch chuẩn ZCR qua các frame |
| 5 | **Mean |amplitude|** | Biên độ tuyệt đối trung bình | $\frac{1}{N}\sum |x[n]|$ |
| 6 | **Std amplitude** | Độ phân tán biên độ | $\sigma(x)$ |

→ Kết quả: **vector 6 chiều** cho mỗi file audio.

> **Nhận xét:** Pipeline A đơn giản, dễ hiểu nhưng chỉ nắm bắt thông tin **miền thời gian**. Nó không phân tích **nội dung tần số** (formant, âm sắc) — vốn rất quan trọng để phân biệt người nói.

In [ ]:
# ── Pipeline A: trích xuất đặc trưng cơ bản ──────────────────────
feat_basic = extract_basic_features(y_raw, sr=SR)

print(f'Vector đặc trưng Pipeline A: {feat_basic.shape[0]} chiều')
print(f'─' * 45)
labels_a = ['RMS mean', 'RMS std', 'ZCR mean', 'ZCR std', 'Mean |amp|', 'Std amp']
for name, val in zip(labels_a, feat_basic):
    print(f'  {name:15s} = {val:.6f}')

---
## 3. Pipeline B — FIR Filter → Pre-emphasis → MFCC (DSP nâng cao)

Pipeline B áp dụng **3 bước xử lý tín hiệu số (DSP)** trước khi trích xuất đặc trưng:

```
Audio thô → [1] Bộ lọc FIR 300-3400Hz → [2] Pre-emphasis → [3] MFCC → Vector 26 chiều
```

Mỗi bước có vai trò cụ thể:
- **Bước 1 (FIR):** Loại bỏ nhiễu tần số thấp (< 300Hz) và tần số cao (> 3400Hz)
- **Bước 2 (Pre-emphasis):** Tăng cường thành phần tần số cao bị suy giảm tự nhiên
- **Bước 3 (MFCC):** Trích xuất đặc trưng mô phỏng cách tai người nghe âm thanh

### 3.1 Bước 1: Bộ lọc FIR bandpass (300–3400 Hz)

**Tại sao lọc 300–3400 Hz?**
- Dải tần này chứa **hầu hết năng lượng giọng nói** (tiêu chuẩn ITU-T telephone speech band)
- Tần số < 300Hz: tiếng ồn nền, rung động, hum điện
- Tần số > 3400Hz: nhiễu, tiếng xì, ít thông tin giọng nói

**Tại sao dùng FIR (không phải IIR)?**
- **Pha tuyến tính** → không làm méo dạng sóng → đặc trưng MFCC chính xác hơn
- **Luôn ổn định** (stable) — không bao giờ phát tán (blow up)
- **Cửa sổ Hamming** → side lobe < -40dB, kiểm soát rò rỉ tần số tốt

Thông số bộ lọc:
- Số tap: **101** (bậc lọc cao → dốc sắc → lọc chính xác)
- Cửa sổ: **Hamming**

In [ ]:
# ── Bước 1: Áp dụng bộ lọc FIR ──────────────────────────────────
coeffs = design_fir(lowcut=300, highcut=3400, sr=SR, numtaps=101)
y_filt = apply_filter(y_raw, coeffs)

print(f'Số hệ số bộ lọc (taps): {len(coeffs)}')
print(f'Signal trước lọc — min: {y_raw.min():.3f}, max: {y_raw.max():.3f}')
print(f'Signal sau lọc   — min: {y_filt.min():.3f}, max: {y_filt.max():.3f}')

# So sánh trước/sau lọc
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

axes[0].plot(np.arange(len(y_raw))/SR, y_raw, linewidth=0.5, color='steelblue')
axes[0].set_ylabel('Biên độ')
axes[0].set_title('Trước lọc FIR (tín hiệu thô)')

axes[1].plot(np.arange(len(y_filt))/SR, y_filt, linewidth=0.5, color='darkorange')
axes[1].set_ylabel('Biên độ')
axes[1].set_xlabel('Thời gian (giây)')
axes[1].set_title('Sau lọc FIR bandpass 300–3400 Hz')

plt.tight_layout()
plt.show()

# Xem phổ tần số trước/sau
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, sig, title, color in zip(axes,
    [y_raw, y_filt],
    ['Phổ tần số — Trước lọc', 'Phổ tần số — Sau lọc FIR'],
    ['steelblue', 'darkorange']):
    
    freqs = np.fft.rfftfreq(len(sig), d=1/SR)
    magnitude = np.abs(np.fft.rfft(sig))
    ax.semilogy(freqs, magnitude, linewidth=0.5, color=color)
    ax.axvline(300, color='red', linestyle='--', alpha=0.7, label='300 Hz')
    ax.axvline(3400, color='green', linestyle='--', alpha=0.7, label='3400 Hz')
    ax.set_xlabel('Tần số (Hz)')
    ax.set_ylabel('Biên độ (log)')
    ax.set_title(title)
    ax.set_xlim(0, SR/2)
    ax.legend()

plt.tight_layout()
plt.show()
print('→ Sau lọc, năng lượng ngoài dải 300–3400 Hz bị loại bỏ rõ rệt.')

### 3.2 Bước 2: Pre-emphasis (Tăng cường tần số cao)

**Vấn đề:** Giọng nói tự nhiên có năng lượng **giảm dần theo tần số** (-6dB/octave). Nghĩa là tần số cao (chứa phụ âm, sibilant) bị "yếu" hơn tần số thấp (chứa nguyên âm).

**Giải pháp:** Áp dụng bộ lọc pre-emphasis bậc 1:

$$y'[n] = y[n] - \alpha \cdot y[n-1] \quad (\alpha = 0.97)$$

**Cách hoạt động:**
- Lấy mẫu hiện tại trừ đi 97% mẫu trước đó
- Hiệu ứng: tăng cường thành phần thay đổi nhanh (tần số cao), giữ nguyên thành phần ổn định (tần số thấp)
- Kết quả: phổ tần số **phẳng hơn** → MFCC trích xuất được nhiều thông tin hơn từ vùng tần số cao

In [ ]:
# ── Bước 2: Pre-emphasis ─────────────────────────────────────────
y_emph = pre_emphasize(y_filt, alpha=0.97)

print(f'Trước pre-emphasis — min: {y_filt.min():.3f}, max: {y_filt.max():.3f}')
print(f'Sau pre-emphasis   — min: {y_emph.min():.3f}, max: {y_emph.max():.3f}')

# So sánh phổ trước/sau pre-emphasis
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, sig, title, color in zip(axes,
    [y_filt, y_emph],
    ['Sau FIR (chưa pre-emphasis)', 'Sau FIR + Pre-emphasis'],
    ['darkorange', 'crimson']):
    
    freqs = np.fft.rfftfreq(len(sig), d=1/SR)
    magnitude = np.abs(np.fft.rfft(sig))
    ax.semilogy(freqs, magnitude, linewidth=0.5, color=color)
    ax.set_xlabel('Tần số (Hz)')
    ax.set_ylabel('Biên độ (log)')
    ax.set_title(title)
    ax.set_xlim(0, SR/2)

plt.tight_layout()
plt.show()
print('→ Sau pre-emphasis, phổ tần số phẳng hơn — tần số cao được tăng cường.')

### 3.3 Bước 3: Trích xuất MFCC (Mel-Frequency Cepstral Coefficients)

MFCC là đặc trưng **phổ biến nhất** trong nhận dạng giọng nói. Nó mô phỏng cách **tai người** cảm nhận âm thanh.

**Quy trình tính MFCC:**
```
Signal → Chia frame (32ms) → FFT từng frame → Mel filterbank → Log → DCT → 13 hệ số MFCC
```

| Bước | Giải thích |
|------|------------|
| **Chia frame** | Cắt signal thành các đoạn nhỏ 32ms (512 mẫu), overlap 16ms (hop=256). Vì giọng nói thay đổi chậm, mỗi frame được coi là "ổn định" |
| **FFT** | Biến đổi Fourier nhanh — chuyển từ miền thời gian sang miền tần số |
| **Mel filterbank** | Áp dụng bộ lọc theo thang Mel — tai người nhạy với tần số thấp hơn tần số cao, thang Mel mô phỏng điều này |
| **Log** | Lấy log năng lượng — tai người cảm nhận âm lượng theo thang logarithm (nhân đôi năng lượng ≈ tăng 3dB) |
| **DCT** | Biến đổi Cosine rời rạc — nén thông tin, loại bỏ tương quan giữa các kênh Mel, giữ 13 hệ số quan trọng nhất |

**Kết quả:** Ma trận MFCC có kích thước **(13 × T)** — 13 hệ số cho mỗi frame, T frame tổng cộng.

Sau đó ta tính **mean** và **std** theo trục thời gian → vector **(26,)** = 13 mean + 13 std.

In [ ]:
# ── Bước 3: Tính ma trận MFCC ────────────────────────────────────
N_FFT = 512       # Kích thước frame = 512 mẫu = 32ms @ 16kHz
HOP_LENGTH = 256  # Bước nhảy = 256 mẫu = 16ms (overlap 50%)
N_MFCC = 13       # Số hệ số MFCC

# Tính MFCC cho cả tín hiệu thô và tín hiệu đã xử lý DSP
mfcc_raw  = librosa.feature.mfcc(y=y_raw,  sr=SR, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
mfcc_emph = librosa.feature.mfcc(y=y_emph, sr=SR, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)

print(f'Ma trận MFCC shape: {mfcc_raw.shape}')
print(f'  → {mfcc_raw.shape[0]} hệ số MFCC × {mfcc_raw.shape[1]} frame thời gian')
print(f'  → Mỗi frame = {N_FFT/SR*1000:.0f}ms, bước nhảy = {HOP_LENGTH/SR*1000:.0f}ms')
print(f'  → Tổng thời gian ≈ {mfcc_raw.shape[1] * HOP_LENGTH / SR:.1f}s')

### Heatmap MFCC — So sánh Raw vs DSP-Enhanced

Heatmap cho thấy **giá trị 13 hệ số MFCC thay đổi theo thời gian**:
- Trục X: thời gian (giây)
- Trục Y: chỉ số hệ số MFCC (0–12)
- Màu sắc: giá trị hệ số (đỏ = cao, xanh = thấp)

Hai heatmap sẽ khác nhau vì Pipeline B đã lọc nhiễu và tăng cường tần số cao trước khi tính MFCC.

In [ ]:
# ── Heatmap MFCC ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, mfcc, title in zip(axes,
                            [mfcc_raw, mfcc_emph],
                            ['MFCC — Tín hiệu thô (Raw)',
                             'MFCC — Sau FIR + Pre-emphasis (Pipeline B)']):
    img = librosa.display.specshow(mfcc, sr=SR, hop_length=HOP_LENGTH,
                                   x_axis='time', ax=ax, cmap='coolwarm')
    ax.set_title(title)
    ax.set_ylabel('Hệ số MFCC')
    ax.set_xlabel('Thời gian (giây)')
    fig.colorbar(img, ax=ax, format='%.0f')

plt.tight_layout()
plt.savefig('../figures/mfcc_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ Pipeline B cho thấy MFCC "sạch" hơn nhờ đã loại nhiễu và cân bằng tần số.')

---
## 4. Tổng hợp: Ma trận MFCC → Vector đặc trưng

Ma trận MFCC có shape **(13 × T)** — quá nhiều chiều và **T thay đổi** tùy theo độ dài audio. Để SVM hoạt động, ta cần vector **cố định kích thước**.

**Cách giải quyết:** Tính **mean** (trung bình) và **std** (độ lệch chuẩn) theo trục thời gian:

```
MFCC matrix (13 × T)  →  mean(13,) + std(13,)  →  vector(26,)
```

| Thành phần | Ý nghĩa |
|------------|---------|
| **13 mean** | "Đặc trưng trung bình" của giọng nói — âm sắc tổng thể, cao độ trung bình |
| **13 std** | "Mức biến thiên" — giọng nói thay đổi nhiều hay ít, phong cách nói |

→ Mỗi file audio 3 giây được biểu diễn bằng **1 vector 26 chiều** — đủ nhỏ để SVM xử lý hiệu quả.

In [ ]:
# ── So sánh vector đặc trưng 2 pipeline ──────────────────────────

# Pipeline A: đặc trưng cơ bản (6 chiều)
feat_a = extract_basic_features(y_raw, sr=SR)

# Pipeline B: MFCC sau FIR + pre-emphasis (26 chiều)
feat_b = extract_mfcc(y_emph, sr=SR)

print('╔══════════════════════════════════════════════════════╗')
print('║        SO SÁNH VECTOR ĐẶC TRƯNG 2 PIPELINE         ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║ Pipeline A (basic):  {feat_a.shape[0]} chiều                       ║')
print(f'║ Pipeline B (MFCC) : {feat_b.shape[0]} chiều                       ║')
print('╚══════════════════════════════════════════════════════╝')

print(f'\n── Pipeline A ({feat_a.shape[0]} chiều) ──')
for name, val in zip(labels_a, feat_a):
    print(f'  {name:15s} = {val:.6f}')

print(f'\n── Pipeline B ({feat_b.shape[0]} chiều) ──')
print('  MFCC mean (13 giá trị):')
print(f'    {feat_b[:13].round(2)}')
print('  MFCC std  (13 giá trị):')
print(f'    {feat_b[13:].round(2)}')

---
## 5. Xây dựng dataset đầy đủ cho cả 2 pipeline

Bây giờ ta áp dụng quy trình trích xuất cho **tất cả file audio** trong `data/index.csv`.

Dataset hiện tại gồm **4 người nói** trong nhóm:

| Speaker | Tên | Số file |
|---------|-----|---------|
| speaker_08 | Cuong | 25 |
| speaker_09 | Quang | 25 |
| speaker_10 | Anne | 25 |
| speaker_11 | Khoa | 25 |

Kết quả lưu vào thư mục `features/`:
- `features_basic.npy` — Pipeline A (n × 6)
- `features_mfcc_filt.npy` — Pipeline B (n × 26)
- `labels.npy` — nhãn speaker_id

In [ ]:
# ── Xem dữ liệu hiện có ──────────────────────────────────────────
df = pd.read_csv('../data/index.csv')
print(f'Tổng số file audio: {len(df)}')
print(f'\nSố file theo speaker:')
print(df.groupby('speaker_name')['filename'].count().to_string())
print(f'\nSpeaker IDs: {sorted(df["speaker_id"].unique())}')

In [ ]:
# ── Trích xuất và lưu features cho cả 2 pipeline ─────────────────
X_basic, X_mfcc, y = save_features('../data/index.csv',
                                    features_dir='../features',
                                    data_dir='../data/raw')

print(f'\n{"="*55}')
print(f'Pipeline A (basic)      : X shape = {X_basic.shape}  ({X_basic.shape[0]} mẫu × {X_basic.shape[1]} đặc trưng)')
print(f'Pipeline B (MFCC + DSP) : X shape = {X_mfcc.shape}  ({X_mfcc.shape[0]} mẫu × {X_mfcc.shape[1]} đặc trưng)')
print(f'Labels                  : y shape = {y.shape}')
print(f'Speakers trong dataset  : {sorted(set(y.tolist()))}')

---
## 6. Trực quan hóa: So sánh khả năng phân tách của 2 Pipeline

Biểu đồ scatter dưới đây vẽ **2 đặc trưng đầu tiên** của mỗi pipeline. Nếu các cụm điểm (cluster) của mỗi speaker **tách rời nhau rõ ràng** → pipeline đó tốt cho phân loại.

- **Pipeline A:** 2 đặc trưng đầu = RMS mean, RMS std (năng lượng)
- **Pipeline B:** 2 đặc trưng đầu = MFCC-1 mean, MFCC-2 mean (âm sắc)

In [ ]:
# ── Scatter plot so sánh 2 pipeline ──────────────────────────────
# Tạo mapping speaker_id → tên
id_to_name = dict(zip(df['speaker_id'], df['speaker_name']))
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pipeline A
ax = axes[0]
for i, spk in enumerate(sorted(set(y.tolist()))):
    mask = y == spk
    name = id_to_name.get(spk, f'Speaker {spk}')
    ax.scatter(X_basic[mask, 0], X_basic[mask, 1],
               label=name, s=50, alpha=0.7, color=colors[i % len(colors)],
               edgecolors='white', linewidth=0.5)
ax.set_title('Pipeline A — Basic Features (6 chiều)', fontsize=12)
ax.set_xlabel('RMS Energy (mean)')
ax.set_ylabel('RMS Energy (std)')
ax.legend()
ax.grid(True, alpha=0.3)

# Pipeline B
ax = axes[1]
for i, spk in enumerate(sorted(set(y.tolist()))):
    mask = y == spk
    name = id_to_name.get(spk, f'Speaker {spk}')
    ax.scatter(X_mfcc[mask, 0], X_mfcc[mask, 1],
               label=name, s=50, alpha=0.7, color=colors[i % len(colors)],
               edgecolors='white', linewidth=0.5)
ax.set_title('Pipeline B — MFCC + DSP (26 chiều)', fontsize=12)
ax.set_xlabel('MFCC-1 mean')
ax.set_ylabel('MFCC-2 mean')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/feature_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ Pipeline B (MFCC) thường cho các cụm tách biệt rõ hơn Pipeline A (basic).')

---
## 7. Tổng kết: So sánh 2 Pipeline

| | Pipeline A (Baseline) | Pipeline B (DSP Enhanced) |
|---|---|---|
| **Kỹ thuật DSP** | Không có | FIR bandpass + Pre-emphasis |
| **Loại đặc trưng** | Miền thời gian (RMS, ZCR) | Miền tần số (MFCC) |
| **Số chiều** | 6 | 26 |
| **Thông tin nắm bắt** | Năng lượng, tốc độ đổi dấu | Âm sắc, formant, đặc điểm thanh quản |
| **Ưu điểm** | Đơn giản, nhanh, dễ hiểu | Phong phú, phân biệt tốt hơn |
| **Nhược điểm** | Thiếu thông tin tần số | Phức tạp hơn, cần DSP |

> **Kỳ vọng:** Pipeline B sẽ cho **accuracy cao hơn** Pipeline A khi train SVM, vì MFCC chứa nhiều thông tin phân biệt người nói hơn.
>
> → Xem kết quả train ở **Notebook 03 (Training)**.